# Step 2 — High-performing gradient boosting validation example — v3.81

This streamlined notebook is **Step 2** of the public notebook workflow.

It assumes **Step 0** has already prepared the environment and generated governed local artifacts. If Step 0 ended in a controlled blocked state, this notebook now reports that state without throwing a traceback.

> **Use boundary:** This is local validation-only experimentation. It is not clinical ECG software, diagnostic software, patient-monitoring software, production ML, benchmark evidence, or proof of deployment fitness.

Recommended order: Step 0 setup → Step 1 narrative walkthrough → Step 2 model validation example.


## Version history and change log

The change log is maintained in reverse chronological order so reviewers see the newest notebook UX changes first while retaining the full version history.

| Version | Change | Rationale |
|---|---|---|
| 3.76 | Moved the public Step 2 notebook to the notebook root path and updated Step 0 / Step 1 links accordingly | Keeps the public workflow in one visible ordered sequence instead of hiding the model example under the nested examples directory |
| 3.76 | Converted Step 2 prerequisite failures into controlled blocked-state UX | Prevents a known Step 0 / PIPE-006 blocked state from turning into a notebook traceback during public review |
| 3.76 | Added explicit `COMMENT MAP` blocks and verbose internal comments to every Python code cell | Makes control flow, mutation boundaries, and validation intent auditable before execution |
| 3.75 | Split environment setup, launch guidance, local artifact preflight, and governed pipeline execution into a dedicated Step 0 setup notebook | Keeps the model notebook focused on validation-only training and evaluation |
| 3.75 | Added a streamlined prerequisite validation suite at the top of the model notebook | Fails elegantly and directs users to Step 0 if required environment or generated artifacts are missing |
| 3.5.2 | Expanded Section 1.1 launcher cells into larger opt-in executable templates with `printf` status updates, step labels, elapsed-time reporting, and clearer no-op behavior when left commented/disabled | Makes setup activity visible during environment preparation while still preventing accidental installs, dependency syncs, or nested JupyterLab launches |
| 3.5.2 | Reordered the change log in reverse chronology | Keeps the most recent notebook governance and UX changes at the top without dropping earlier version history |
| 3.5.1 | Converted the local terminal launcher from markdown fenced code blocks into opt-in commented `%%bash` code cells with explicit `**INSTRUCTIONS**` guidance | Keeps copy/paste setup commands in executable notebook cells while preventing accidental environment mutation or nested JupyterLab launches |
| 3.5 | Added launch and environment options for local JupyterLab, GitHub Codespaces, and Colab-style runtimes | Gives first-time reviewers practical paths for opening and preparing the notebook environment |
| 3.5 | Added optional environment bootstrap, runtime diagnostics, and guarded launcher-script creation cells | Improves reviewer onboarding without automatically modifying the repository or weakening artifact governance |
| 3.0 | Added defensive preflight checks for manifestless raw files, interrupted acquisition directories, and stale generated artifacts | Converts common local reproducibility failures into actionable remediation guidance before artifact loading |
| 3.0 | Added targeted pipeline-failure interpretation for acquisition-manifest errors and inconsistent window-shard identity errors | Helps users recover from partial local runs without hiding the underlying reproducibility guardrails |
| 3.0 | Added safe clean-rebuild remediation commands while avoiding automatic destructive cleanup | Keeps remediation explicit and user-controlled |
| 2.0 | Added first-run prerequisite guidance, governed pipeline command, and `**CODE CELL FUNCTION**` tags before code cells | Made the notebook more usable for reviewers opening it from a fresh clone |
| 2.0 | Added clearer comments across setup, artifact loading, split validation, model fitting, metrics, and lineage cells | Improved scanability and reduced hidden assumptions |
| 2.0 | Cleared execution counts and outputs | Prevented committed local execution state, generated metrics, and prior failures from appearing in the public notebook |
| 1.0 | Initial public high-performing gradient boosting validation notebook | Demonstrated a fixed classical ML validation-only example using generated pipeline artifacts |

## Version history and change log

| Version | Change |
|---:|---|
| 3.81 | Removed stale non-throwing prerequisite language from the public Step 2 description so the notebook contract matches strict validation-example behavior. |
| 3.80 | Restored strict validation-example semantics. Step 2 no longer treats missing Step 0 artifacts or missing modeling dependencies as a successful skipped notebook path; it fails fast until the public workflow can actually load governed artifacts, train, and evaluate on validation. |
| 3.77 | Fixed blocked-mode rendering when modeling dependencies are missing. Step 2 now defines a `display_markdown` helper before optional modeling imports, so missing `sklearn` reports as controlled notebook UX instead of `TypeError: NoneType object is not callable`. |


## 0. Stop — complete Step 0 first

<div style="border: 4px solid #b00020; background: #fff1f1; color: #7a0016; padding: 18px; font-size: 1.25rem; font-weight: 700; line-height: 1.45;">
STOP: DO NOT START MODEL TRAINING UNTIL STEP 0 HAS COMPLETED.<br><br>
Run <code>00-environment-setup-and-artifact-generation.ipynb</code> first if you have not already generated the local governed artifacts.
</div>

Step 0 handles environment setup, launch options, local artifact preflight, governed pipeline execution, and post-pipeline verification.

Open Step 0 here:

[`00-environment-setup-and-artifact-generation.ipynb`](00-environment-setup-and-artifact-generation.ipynb)

This notebook still includes a lightweight prerequisite/status suite below. If Step 0 has not fully generated model-ready artifacts, the suite reports a controlled blocked state instead of allowing a later cryptic file-loading error.

Then read the supported modernization walkthrough if you want the repository lifecycle context:

[`01-narrative-walkthrough.ipynb`](01-narrative-walkthrough.ipynb)


> **CODE CELL FUNCTION**
>
> Run a lightweight prerequisite/status suite before importing modeling dependencies or loading generated artifacts. This cell is strict: Step 2 stops when Step 0 has not produced model-ready artifacts or when the active kernel cannot import required modeling dependencies.


In [ ]:
# CODE CELL FUNCTION:
# Enforce Step 0 artifact readiness before the model validation example runs.
#
# COMMENT MAP:
# - Treat Step 2 as a real validation example, not a graceful-skip report.
# - Require the uv-backed notebook kernel to expose all modeling dependencies.
# - Require Step 0 status to be complete and verify the exact artifacts Step 2 loads.
# - Fail fast when PIPE-006 or any other Step 0 failure prevents model-ready artifacts.
# - Keep the protected test partition unopened; only train/validation readiness is checked.
# - Initialize shared paths only after the prerequisite contract is satisfied.

from __future__ import annotations

import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

STEP_0_NOTEBOOK = "notebooks/00-environment-setup-and-artifact-generation.ipynb"
STEP_1_NOTEBOOK = "notebooks/01-narrative-walkthrough.ipynb"
STEP2_READY = False
STEP2_BLOCKERS: list[str] = []
STEP2_WARNINGS: list[str] = []
STEP2_CONTEXT: dict[str, object] = {
    "repository_root": None,
    "dataset_index": None,
    "run_id": None,
    "train_validation_artifacts_ready": False,
    "test_partition_opened": False,
}


def discover_repository_root(start: Path) -> Path:
    """Return the nearest repository root or raise before any model work begins."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Repository root was not found. Open Step 2 from inside the cloned repository.")


def load_json_object(path: Path) -> dict[str, Any]:
    """Load one JSON object and reject missing, malformed, or non-object payloads."""
    if not path.is_file():
        raise RuntimeError(f"Required artifact is missing: {path}")
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Invalid JSON artifact: {path}") from exc
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected JSON object artifact: {path}")
    return value


def relative_to_root(path: Path, root: Path) -> str:
    """Render a repository-relative path when possible for readable failure output."""
    try:
        return path.relative_to(root).as_posix()
    except ValueError:
        return path.as_posix()


def require_modules(module_names: list[str]) -> None:
    """Require notebook/modeling dependencies before any validation cells run."""
    missing = [name for name in module_names if importlib.util.find_spec(name) is None]
    if missing:
        raise RuntimeError(
            "Step 2 cannot run the validation example because the active kernel is missing dependencies: "
            + ", ".join(missing)
            + ". Run Step 0 environment bootstrap and select the uv-backed notebook kernel."
        )


def require_step0_complete(root: Path) -> dict[str, Any]:
    """Require strict Step 0 success instead of accepting blocked status as readiness."""
    status_path = root / "notebooks" / "local" / "step0-pipeline-status.json"
    status = load_json_object(status_path)
    if status.get("status") != "complete":
        raise RuntimeError(
            "Step 2 requires Step 0 to complete model-ready artifact generation. Step 0 did not complete.\n"
            f"Status: {status.get('status')}\n"
            f"Reason: {status.get('reason')}\n"
            f"Message: {status.get('message')}\n"
            f"Log file: {status.get('log_file')}"
        )
    return status


def newest_dataset_index(root: Path) -> Path:
    """Return the newest generated dataset index produced by Step 0 or raise."""
    candidates = sorted(
        (root / "data" / "processed" / "runs").glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise RuntimeError(
            "Step 2 cannot run because Step 0 did not produce "
            "data/processed/runs/<run-id>/dataset-index.json."
        )
    return candidates[-1]


root = discover_repository_root(Path.cwd())
STEP2_CONTEXT["repository_root"] = str(root)

require_modules(["numpy", "sklearn", "matplotlib", "IPython", "ecg_anomaly_detection"])
if shutil.which("uv") is None:
    raise RuntimeError("`uv` is not available on PATH; Step 2 requires the repository uv environment.")
cli_check = subprocess.run(
    ["uv", "run", "ecg-data", "--help"],
    cwd=root,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)
if cli_check.returncode != 0:
    raise RuntimeError("`uv run ecg-data --help` failed; Step 2 is not running in the supported repo environment.\n" + cli_check.stderr[-1000:])

step0_status = require_step0_complete(root)
index_path = newest_dataset_index(root)
run_id = index_path.parent.name
run_artifact_root = root / "artifacts" / "runs" / run_id
required_artifacts = {
    "dataset_index": index_path,
    "split_manifest": run_artifact_root / "split.json",
    "split_quality": run_artifact_root / "split_quality_summary.json",
}
missing = [f"{label}: {relative_to_root(path, root)}" for label, path in required_artifacts.items() if not path.is_file()]
if missing:
    raise RuntimeError("Step 2 cannot run because required Step 0 artifacts are missing:\n" + "\n".join(missing))

index_probe = load_json_object(index_path)
partitions = index_probe.get("partitions")
if not isinstance(partitions, dict) or set(partitions) != {"train", "validation", "test"}:
    raise RuntimeError("Dataset index must expose exactly train, validation, and test partitions.")
for partition_name in ("train", "validation"):
    partition = partitions.get(partition_name, {})
    shards = partition.get("shards", [])
    if not isinstance(shards, list) or not shards:
        raise RuntimeError(f"Dataset index partition `{partition_name}` must expose at least one shard.")
    missing_shards: list[str] = []
    for shard in shards:
        relative = shard.get("path") or shard.get("relative_path")
        if not relative or not (root / relative).is_file():
            missing_shards.append(str(relative))
    if missing_shards:
        raise RuntimeError(f"Missing {partition_name} shard files:\n" + "\n".join(missing_shards[:20]))

STEP2_READY = True
STEP2_CONTEXT.update({
    "dataset_index": relative_to_root(index_path, root),
    "run_id": run_id,
    "train_validation_artifacts_ready": True,
    "test_partition_opened": False,
})

{
    "step_2_ready": STEP2_READY,
    "step_0_status": step0_status.get("status"),
    "run_id": run_id,
    "dataset_index": relative_to_root(index_path, root),
    "split_manifest": relative_to_root(run_artifact_root / "split.json", root),
    "split_quality": relative_to_root(run_artifact_root / "split_quality_summary.json", root),
    "test_partition_opened": False,
}


## 1. Model notebook flow

After the Step 0 prerequisite suite passes, this notebook does only the streamlined validation workflow:

1. discover repository-relative generated artifacts
2. validate grouped train/validation/test split metadata
3. load train and validation waveform windows
4. train one fixed `HistGradientBoostingClassifier`
5. report validation-only metrics
6. summarize lineage and limitations

The protected test partition is indexed for governance context but is not opened, scored, or reported.


> **CODE CELL FUNCTION**
>
> Import modeling dependencies only after the readiness/status cell has run. Missing imports are converted into additional blocked-state messages so the notebook can still finish with actionable Step 0 guidance.


In [ ]:
# COMMENT MAP:
# - Import display helpers and modeling dependencies only after the strict prerequisite gate passes.
# - Do not catch ImportError here; Step 2 is supposed to fail if the environment is wrong.
# - Keep imports explicit so reviewers can see the modeling stack used by the example.
# - Preserve validation-only scope: metrics are imported for validation, not protected test evaluation.

import hashlib
from collections import Counter
from typing import Any

from IPython.display import Markdown as _Markdown
from IPython.display import display as _display
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_sample_weight


def display_markdown(text: str) -> None:
    """Render Markdown in notebook environments without hiding dependency failures."""
    _display(_Markdown(text))

{
    "imports_ready": True,
    "step_2_ready": STEP2_READY,
    "modeling_stack": ["numpy", "matplotlib", "sklearn"],
}


> **CODE CELL FUNCTION**
>
> Resolve repository paths through the installed package when available. If Step 2 is blocked, this cell reports the blocked state and avoids package-backed assertions.


In [ ]:

# COMMENT MAP:
# - Prefer the package-backed RepositoryPaths helper when the environment is ready.
# - Do not assert or crash when Step 0 has not prepared the kernel; report blocked state instead.
# - Keep `root` stable from the prereq cell so later repository-relative diagnostics still work.
# - Avoid mutating files; this is environment orientation only.

if not STEP2_READY:
    # The prereq/import cells already captured the reason. Repeating it here keeps execution linear.
    {
        "repository_paths_ready": False,
        "reason": "Step 2 is blocked before package-backed path resolution.",
        "blockers": STEP2_BLOCKERS,
    }
else:
    from ecg_anomaly_detection.config import RepositoryPaths

    paths = RepositoryPaths.discover()
    root = paths.root

    # Convert unexpected path mismatch into a blocker instead of an assertion wall.
    if not (root / "pyproject.toml").is_file():
        STEP2_READY = False
        STEP2_BLOCKERS.append("Repository root must contain pyproject.toml.")
        {
            "repository_paths_ready": False,
            "blockers": STEP2_BLOCKERS,
        }
    else:
        {
            "repository_root": str(root),
            "python": sys.executable,
            "package_backed": True,
        }


> **CODE CELL FUNCTION**
>
> Define JSON-loading, local-state preflight, remediation-message, and artifact-discovery helpers with actionable missing-artifact messages.


In [ ]:

# COMMENT MAP:
# - Define helper functions for JSON loading, artifact readiness, and path remediation.
# - Keep helpers side-effect free except for reading local files.
# - Allow helpers to raise precise exceptions only when Step 2 is already ready and a contract is broken.
# - Keep remediation text focused on Step 0 because this notebook should not clean or rebuild artifacts.
# - Maintain protected-test discipline by requiring only train/validation artifacts downstream.

def load_json(path: Path) -> dict[str, Any]:
    """Load a JSON object from disk and reject non-object payloads.

    This helper is used after readiness checks pass. A non-object payload means the
    generated artifact contract is invalid and should be surfaced clearly.
    """
    with path.open(encoding="utf-8") as source:
        value = json.load(source)
    if not isinstance(value, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return value


def local_clean_rebuild_message(*, repository_root: Path) -> str:
    """Return safe remediation guidance for rebuilding ignored local pipeline outputs.

    The command text is displayed only as guidance. This notebook never runs destructive
    cleanup automatically.
    """
    return (
        "Suggested remediation from the repository root:\n\n"
        "Run Step 0 first:\n"
        "  notebooks/00-environment-setup-and-artifact-generation.ipynb\n\n"
        "Step 0 owns safe local cleanup, governed acquisition, pipeline execution, "
        "and blocked-state diagnosis. Do not commit generated `data/` or `artifacts/` outputs."
    )


def missing_artifact_message(missing_path: Path, *, repository_root: Path) -> str:
    """Build a clear remediation message for missing generated artifacts."""
    relative = missing_path.relative_to(repository_root) if missing_path.is_relative_to(repository_root) else missing_path
    return (
        f"Required generated artifact is missing: {relative}\n\n"
        "Generated pipeline artifacts are intentionally ignored by Git, so a fresh clone will not include them.\n"
        "Complete Step 0 first: `notebooks/00-environment-setup-and-artifact-generation.ipynb`.\n\n"
        + local_clean_rebuild_message(repository_root=repository_root)
    )


def require_file(path: Path, *, repository_root: Path, purpose: str) -> Path:
    """Return a required file path or raise a precise artifact-contract error."""
    if not path.is_file():
        raise FileNotFoundError(purpose + "\n\n" + missing_artifact_message(path, repository_root=repository_root))
    return path


def latest_dataset_index(repository_root: Path) -> Path:
    """Return the newest generated dataset index for local validation experimentation."""
    processed_runs_dir = repository_root / "data" / "processed" / "runs"
    candidates = sorted(
        processed_runs_dir.glob("*/dataset-index.json"),
        key=lambda path: path.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            "No generated dataset index was found.\n\n"
            + missing_artifact_message(processed_runs_dir / "<run-id>" / "dataset-index.json", repository_root=repository_root)
        )
    return candidates[-1]


{
    "helper_functions_loaded": True,
    "side_effects": "none; helpers only read local artifacts when called",
    "step_2_ready": STEP2_READY,
}


> **CODE CELL FUNCTION**
>
> Resolve the current generated run, required manifests, and optional baseline metrics.


In [ ]:

# COMMENT MAP:
# - Select the latest generated dataset index only when prerequisites are ready.
# - Keep all run-specific artifact paths tied to the same run identifier.
# - Treat baseline metrics as optional because Step 2 can report validation metrics without them.
# - Avoid undefined variables in blocked mode by initializing placeholders.

index_path = None
run_id = None
run_artifact_root = None
split_path = None
split_quality_path = None
baseline_metrics_path = None
index = {}
split_manifest = {}
split_quality = {}
baseline_metrics_available = False

if not STEP2_READY:
    # Step 0 may have ended in a controlled blocked state such as PIPE-006.
    # In that case, Step 2 should complete as a readable blocked-state report.
    raise RuntimeError("Step 2 strict prerequisite gate failed before artifact loading: " + "; ".join(STEP2_BLOCKERS))
else:
    # The newest dataset index is treated as the local run selected for this notebook session.
    index_path = latest_dataset_index(root)
    run_id = index_path.parent.name

    # Related run artifacts are expected to share the same run identifier.
    run_artifact_root = root / "artifacts" / "runs" / run_id
    split_path = run_artifact_root / "split.json"
    split_quality_path = run_artifact_root / "split_quality_summary.json"
    baseline_metrics_path = run_artifact_root / "evaluation" / "validation-metrics.json"

    # Required artifacts are loaded or rejected with actionable messages.
    index = load_json(require_file(index_path, repository_root=root, purpose="Dataset index is required before loading train/validation windows."))
    split_manifest = load_json(require_file(split_path, repository_root=root, purpose="Split manifest is required before validating grouped partition lineage."))
    split_quality = load_json(require_file(split_quality_path, repository_root=root, purpose="Split quality summary is required before describing split validation status."))

    # Baseline metrics are useful for comparison but not required for this notebook to run.
    baseline_metrics_available = baseline_metrics_path.is_file()

{
    "step_2_ready": STEP2_READY,
    "run_id": run_id,
    "dataset_index": str(index_path.relative_to(root)) if index_path else None,
    "split_manifest": str(split_path.relative_to(root)) if split_path else None,
    "split_quality": str(split_quality_path.relative_to(root)) if split_quality_path else None,
    "baseline_metrics_available": baseline_metrics_available,
}


## 3. Validate grouped split assumptions

This check verifies the high-level grouping contract from the dataset index before any waveform arrays are opened.

It confirms:

- expected partition names exist
- subjects do not cross partitions
- records do not cross partitions
- the protected test partition remains indexed for lineage but is not opened for scoring


> **CODE CELL FUNCTION**
>
> Validate dataset-index partition structure and confirm train/validation/test grouping boundaries.


In [ ]:

# COMMENT MAP:
# - Validate grouped split assumptions only after generated manifests are loaded.
# - Confirm train, validation, and protected test partitions exist in metadata.
# - Check subject and record disjointness without opening protected test waveform arrays.
# - Keep blocked mode non-throwing so public execution can finish with a clear status.

expected_partitions = {"train", "validation", "test"}
partitions = {}
subject_sets = {}
record_sets = {}
split_status = "not evaluated"

if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so split validation must not run.")
else:
    partitions = index.get("partitions")

    # The governed split contract should expose exactly these three partitions.
    if not isinstance(partitions, dict) or set(partitions) != expected_partitions:
        raise ValueError("Dataset index must contain exactly train, validation, and test partitions")


    def partition_subjects(name: str) -> set[str]:
        """Extract subject identifiers for one partition."""
        value = partitions[name]
        subjects = value.get("subject_ids", [])
        if not isinstance(subjects, list):
            raise ValueError(f"Partition {name} has invalid subject_ids")
        return set(str(subject) for subject in subjects)


    def partition_records(name: str) -> set[str]:
        """Extract record identifiers from one partition's shard descriptors."""
        shards = partitions[name].get("shards", [])
        if not isinstance(shards, list):
            raise ValueError(f"Partition {name} has invalid shards")
        records = {str(shard.get("record_id")) for shard in shards}
        if "" in records or "None" in records:
            raise ValueError(f"Partition {name} has invalid record IDs")
        return records


    subject_sets = {name: partition_subjects(name) for name in expected_partitions}
    record_sets = {name: partition_records(name) for name in expected_partitions}

    # Pairwise disjointness checks guard against accidental subject or record leakage.
    for left in expected_partitions:
        for right in expected_partitions - {left}:
            if subject_sets[left] & subject_sets[right]:
                raise ValueError(f"Subject leakage detected between {left} and {right}")
            if record_sets[left] & record_sets[right]:
                raise ValueError(f"Record leakage detected between {left} and {right}")

    split_status = split_quality.get("status", "not available")

{
    "step_2_ready": STEP2_READY,
    "schema_version": index.get("schema_version") if index else None,
    "split": f"{index.get('split_name')}@{index.get('split_version')}" if index else None,
    "split_quality_status": split_status,
    "train_records": len(record_sets.get("train", set())),
    "validation_records": len(record_sets.get("validation", set())),
    "test_records_indexed_but_not_opened": len(record_sets.get("test", set())),
}


## 4. Load raw waveform windows

Only train and validation shard descriptors are resolved and opened.

The protected test partition remains indexed for lineage but unopened.

The features are raw waveform window rows stored in each record-level NPZ shard. No FFT augmentation, engineered feature expansion, local checkpoints, or exploratory outputs are used.


> **CODE CELL FUNCTION**
>
> Define file integrity helpers and load only train/validation waveform windows from indexed NPZ shards.


In [ ]:

# COMMENT MAP:
# - Load waveform windows only for train and validation partitions.
# - Refuse protected test loading at the helper boundary.
# - Validate shard hashes, array shapes, finite values, targets, and record lineage.
# - Initialize empty placeholders in blocked mode so later cells can skip safely.

X_train = None
y_train = None
train_records = []
X_validation = None
y_validation = None
validation_records = []


def sha256_file(path: Path) -> str:
    """Compute a SHA-256 digest for an indexed shard when expected hashes are present."""
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_indexed_file(shard: dict[str, Any], *, partition: str) -> Path:
    """Resolve and validate one shard path from the dataset index."""
    relative_path = shard.get("path") or shard.get("relative_path")
    if not isinstance(relative_path, str) or not relative_path:
        raise ValueError(f"Shard in {partition} partition is missing a relative path")

    shard_path = root / relative_path
    require_file(shard_path, repository_root=root, purpose=f"Window shard for {partition} partition is required before loading waveform arrays.")

    expected_sha256 = shard.get("sha256")
    if expected_sha256:
        observed_sha256 = sha256_file(shard_path)
        if observed_sha256 != expected_sha256:
            raise ValueError(f"SHA-256 mismatch for indexed shard: {shard_path.relative_to(root)}")
    return shard_path


def load_partition(name: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load all waveform windows and integer targets for one non-test partition."""
    if name == "test":
        raise ValueError("Protected test partition must not be opened by this notebook")

    matrices: list[np.ndarray] = []
    targets: list[np.ndarray] = []
    records: list[str] = []
    partition = partitions[name]

    for shard in partition["shards"]:
        shard_path = resolve_indexed_file(shard, partition=name)
        expected_record = str(shard["record_id"])

        # Each record-level shard should contain windows, targets, and record lineage arrays.
        with np.load(shard_path, allow_pickle=False) as artifact:
            windows = np.asarray(artifact["windows"])
            labels = np.asarray(artifact["target_values"])
            record_ids = np.asarray(artifact["record_ids"])

        # Validate shape, dtype, finiteness, label alignment, and lineage before concatenation.
        if windows.ndim != 2 or windows.dtype.kind != "f" or not np.isfinite(windows).all():
            raise ValueError(f"Invalid finite floating-point window matrix: {shard_path.relative_to(root)}")
        if labels.ndim != 1 or labels.dtype.kind not in {"i", "u"} or len(labels) != len(windows):
            raise ValueError(f"Invalid target vector: {shard_path.relative_to(root)}")
        if set(record_ids.tolist()) != {expected_record}:
            raise ValueError(f"Record lineage mismatch: {shard_path.relative_to(root)}")

        matrices.append(windows.astype(np.float64, copy=False))
        targets.append(labels.astype(np.int64, copy=False))
        records.append(expected_record)

    features = np.concatenate(matrices, axis=0)
    labels = np.concatenate(targets)

    # Compare loaded counts against the dataset index to catch incomplete or mismatched artifacts.
    expected_counts = {str(k): v for k, v in sorted(Counter(labels.tolist()).items())}
    if expected_counts != partition.get("target_value_counts"):
        raise ValueError(f"Loaded {name} targets do not match dataset index counts")
    if features.shape[0] != partition.get("window_count"):
        raise ValueError(f"Loaded {name} window count does not match dataset index")
    return features, labels, records


if not STEP2_READY:
    display_markdown("**Waveform loading skipped.** Step 2 is blocked until Step 0 produces train/validation shards.")
else:
    X_train, y_train, train_records = load_partition("train")
    X_validation, y_validation, validation_records = load_partition("validation")

{
    "step_2_ready": STEP2_READY,
    "feature_source": "raw waveform windows only" if STEP2_READY else None,
    "window_samples": int(index["window_samples"]) if STEP2_READY else None,
    "train_shape": tuple(int(value) for value in X_train.shape) if STEP2_READY else None,
    "validation_shape": tuple(int(value) for value in X_validation.shape) if STEP2_READY else None,
    "train_counts": {str(k): int(v) for k, v in Counter(y_train.tolist()).items()} if STEP2_READY else {},
    "validation_counts": {str(k): int(v) for k, v in Counter(y_validation.tolist()).items()} if STEP2_READY else {},
    "test_partition_opened": False,
}


## 5. Train fixed tuned gradient boosting model

This example intentionally uses one fixed configuration derived from prior local experimentation.

It is not a hyperparameter search notebook and does not claim this configuration is universally optimal.

Balanced sample weights are applied during fitting to reduce the effect of class imbalance in the training partition.


> **CODE CELL FUNCTION**
>
> Fit one fixed `HistGradientBoostingClassifier` on the train partition using balanced sample weights.


In [ ]:

# COMMENT MAP:
# - Train one fixed HistGradientBoostingClassifier only when train/validation arrays are loaded.
# - Keep the hyperparameters fixed; this is not a search loop or leaderboard optimization.
# - Use balanced sample weights to reduce majority-class dominance during local validation.
# - Do not save a model artifact; the repository artifact boundary remains clean.

model = None

if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so model fitting must not run.")
else:
    # Fixed model configuration: no search loop, no leaderboard tuning, no protected-test feedback.
    model = HistGradientBoostingClassifier(
        learning_rate=0.015,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=0.1,
        max_iter=450,
        random_state=0,
    )

    # Balanced weights reduce the dominance of majority classes during training.
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

    # The model is trained in memory only. No trained model is written to artifacts/.
    model.fit(X_train, y_train, sample_weight=sample_weight)

{
    "step_2_ready": STEP2_READY,
    "estimator": model.__class__.__name__ if model is not None else None,
    "learning_rate": model.learning_rate if model is not None else None,
    "max_leaf_nodes": model.max_leaf_nodes if model is not None else None,
    "min_samples_leaf": model.min_samples_leaf if model is not None else None,
    "l2_regularization": model.l2_regularization if model is not None else None,
    "max_iter": model.max_iter if model is not None else None,
    "sample_weighting": "balanced" if model is not None else None,
    "model_saved_as_repository_artifact": False,
}


## 6. Validation-only metrics

The next cell scores only the validation partition.

These are observed local validation-only results for the grouped validation partition. They are not benchmark evidence, clinical evidence, protected-test evidence, or production model performance.


> **CODE CELL FUNCTION**
>
> Predict validation labels and display precision, recall, F1, support, and accuracy as a Markdown table.


In [ ]:

# COMMENT MAP:
# - Evaluate only the validation partition.
# - Derive metric labels from train plus validation labels to keep zero-support classes visible.
# - Render a Markdown table for public readability.
# - Avoid opening or referencing protected test labels.

validation_predictions = None
classes = ()
accuracy = None
macro_precision = None
macro_recall = None
macro_f1 = None
weighted_precision = None
weighted_recall = None
weighted_f1 = None

if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; strict prerequisite gate did not pass, so validation metrics must not run.")
else:
    # Validation is the only evaluation partition opened by this notebook.
    validation_predictions = model.predict(X_validation)

    # Classes are derived from train plus validation labels so zero-support edge cases remain visible.
    classes = tuple(int(value) for value in sorted(set(y_train.tolist()) | set(y_validation.tolist())))

    precision, recall, f1, support = precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        labels=list(classes),
        zero_division=0,
    )
    accuracy = accuracy_score(y_validation, validation_predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_validation,
        validation_predictions,
        average="weighted",
        zero_division=0,
    )

    rows = [
        "| Class | Precision | Recall | F1 | Support |",
        "|---:|---:|---:|---:|---:|",
    ]
    for label, p, r, score, count in zip(classes, precision, recall, f1, support, strict=True):
        rows.append(f"| {label} | {p:.4f} | {r:.4f} | {score:.4f} | {int(count)} |")
    rows.extend([
        f"| **Macro avg** | **{macro_precision:.4f}** | **{macro_recall:.4f}** | **{macro_f1:.4f}** | **{len(y_validation)}** |",
        f"| **Weighted avg** | **{weighted_precision:.4f}** | **{weighted_recall:.4f}** | **{weighted_f1:.4f}** | **{len(y_validation)}** |",
        f"| **Accuracy** |  |  | **{accuracy:.4f}** | **{len(y_validation)}** |",
    ])

    display_markdown("\n".join(rows))


## 7. Confusion matrix

Rows are true validation labels. Columns are predicted labels.

This figure is a notebook visualization only. It is not saved as a repository artifact by this notebook.


> **CODE CELL FUNCTION**
>
> Display a validation-only confusion matrix for local inspection.


In [ ]:

# COMMENT MAP:
# - Plot the validation-only confusion matrix when predictions exist.
# - Keep the figure local to the notebook; no plot artifact is written to the repository.
# - Annotate each cell so the chart remains interpretable in static notebook views.
# - Skip cleanly when Step 2 is blocked before model fitting.

if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; validation predictions do not exist, so confusion matrix must not run.")
else:
    # Confusion matrix uses validation labels and validation predictions only.
    matrix = confusion_matrix(y_validation, validation_predictions, labels=list(classes))

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(matrix)
    ax.set_title("Validation-only confusion matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(len(classes)), classes)
    ax.set_yticks(range(len(classes)), classes)

    # Annotate each cell so the plot remains readable without hovering or external files.
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, str(int(matrix[row, column])), ha="center", va="center")

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


## 8. Baseline comparison when available

If the same pipeline run includes deterministic baseline validation metrics, this cell compares the local gradient boosting validation result to that repository baseline artifact.

Missing baseline metrics are handled explicitly so the notebook remains usable when only processed dataset artifacts exist.


> **CODE CELL FUNCTION**
>
> Load optional repository baseline validation metrics and compare them with the in-memory gradient boosting result.


In [ ]:

# COMMENT MAP:
# - Compare against optional baseline validation metrics only when available.
# - Treat missing baseline metrics as an informational condition, not a failure.
# - Keep comparison strictly validation-only.
# - Skip cleanly when Step 2 is blocked before model metrics exist.

def baseline_summary(path: Path) -> dict[str, float] | None:
    """Load optional baseline validation metrics from the governed pipeline run."""
    if not path or not path.is_file():
        return None
    metrics = load_json(path)
    macro = metrics.get("macro_average", {})
    return {
        "accuracy": float(metrics.get("accuracy")),
        "macro_precision": float(macro.get("precision")),
        "macro_recall": float(macro.get("recall")),
        "macro_f1": float(macro.get("f1")),
    }


if not STEP2_READY:
    raise RuntimeError("Step 2 is not ready; validation metrics do not exist, so baseline comparison must not run.")
else:
    baseline = baseline_summary(baseline_metrics_path)
    gradient_boosting = {
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
    }

    if baseline is None:
        display_markdown("Baseline validation metrics were not found for this run, so no baseline comparison is shown.")
    else:
        rows = [
            "| Model | Accuracy | Macro precision | Macro recall | Macro F1 |",
            "|---|---:|---:|---:|---:|",
            f"| Repository deterministic baseline | {baseline['accuracy']:.4f} | {baseline['macro_precision']:.4f} | {baseline['macro_recall']:.4f} | {baseline['macro_f1']:.4f} |",
            f"| Fixed HistGradientBoosting local example | {gradient_boosting['accuracy']:.4f} | {gradient_boosting['macro_precision']:.4f} | {gradient_boosting['macro_recall']:.4f} | {gradient_boosting['macro_f1']:.4f} |",
        ]
        display_markdown("\n".join(rows))


## 9. Lineage and limitations

The trained estimator in this notebook is an in-memory local experiment. It is not saved as a supported model artifact, not added to the run manifest, and not promoted to benchmark status.

Interpretation boundaries:

- Inputs are generated local pipeline artifacts under ignored `data/` and `artifacts/` paths.
- Features are raw waveform windows only.
- Fitting uses the train partition only.
- Evaluation uses the validation partition only.
- The protected test partition remains unopened and unreported.
- Observed values are local experiment results, not benchmark evidence.
- Results are not clinical evidence, diagnostic evidence, deployment fitness, or medical utility.
- LightGBM remains an optional future local optimization candidate, not a default public notebook dependency.


> **CODE CELL FUNCTION**
>
> Summarize run lineage, model boundaries, validation-only metrics, and claim limitations in a compact dictionary.


In [ ]:

# COMMENT MAP:
# - Summarize lineage and claim boundaries whether Step 2 is ready or blocked.
# - Avoid undefined variables by using guarded expressions for run-specific values.
# - Preserve the explicit non-benchmark, non-clinical, non-production claim boundary.
# - Confirm again that the protected test partition was not opened.

if STEP2_READY:
    summary = {
        "step_2_status": "completed_validation_example",
        "run_id": run_id,
        "dataset_index": str(index_path.relative_to(root)),
        "split": f"{index.get('split_name')}@{index.get('split_version')}",
        "mapping": f"{index.get('mapping_name')}@{index.get('mapping_version')}",
        "windowing": f"{index.get('window_config_name')}@{index.get('window_config_version')}",
        "sample_rate_hz": index.get("sample_rate_hz"),
        "channel": {"index": index.get("channel_index"), "name": index.get("channel_name")},
        "features": "raw waveform windows only",
        "estimator": "HistGradientBoostingClassifier",
        "training_partition": "train",
        "evaluation_partition": "validation",
        "test_partition_opened": False,
        "model_saved_as_repository_artifact": False,
        "validation_accuracy": float(accuracy),
        "validation_macro_f1": float(macro_f1),
        "claim_boundary": "local validation-only experiment; not benchmark, clinical, or production evidence",
    }
else:
    summary = {
        "step_2_status": "blocked_before_model_training",
        "blockers": STEP2_BLOCKERS,
        "warnings": STEP2_WARNINGS,
        "context": STEP2_CONTEXT,
        "training_partition": None,
        "evaluation_partition": None,
        "test_partition_opened": False,
        "model_saved_as_repository_artifact": False,
        "claim_boundary": "blocked local validation-only example; no benchmark, clinical, or production evidence",
        "required_next_step": STEP_0_NOTEBOOK,
    }

summary
